# Make Lapse Labels

Coco Yu  
August 19, 2024

## Code Status

## Setup

Chunk Defaults

In [ ]:
knitr::opts_chunk$set(attr.output='style="max-height: 500px;"')

Conflicts

In [ ]:
options(conflicts.policy = "depends.ok")

In [ ]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.3     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.3     ✔ tibble    3.2.1
✔ lubridate 1.9.2     ✔ tidyr     1.3.0
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

here() starts at C:/Users/jyu274/Desktop/github/study_messages

Source Functions

In [ ]:
devtools::source_url("https://github.com/jjcurtin/lab_support/blob/main/format_path.R?raw=true")

ℹ SHA-1 hash of file is "a58e57da996d1b70bb9a5b58241325d6fd78890f"

ℹ SHA-1 hash of file is "c045eee2655a18dc85e715b78182f176327358a7"

ℹ SHA-1 hash of file is "def6ce26ed7b2493931fde811adff9287ee8d874"

Absolute Paths

In [ ]:
path_shared <- format_path(str_c("studydata/risk/data_processed/shared"))
path_messages <- format_path(str_c("studydata/risk/data_processed/messages"))

## Read in Data

need further cleaning on lapses: for now, NA if NA in start_time

In [ ]:
visit_dates <- read_csv(here(path_shared, "visit_dates.csv"),
                        col_types = cols(subid = "d", start_study = "c", 
                                         end_study = "c")) |> 
  select(subid, start_study, end_study) |> 
  glimpse()

Rows: 207
Columns: 3
$ subid       <dbl> 1, 2, 3, 5, 6, 7, 9, 10, 11, 15, 16, 17, 18, 19, 20, 21, 2…
$ start_study <chr> "2017-03-02", "2017-03-24", "2017-03-22", "2017-06-20", "2…
$ end_study   <chr> "2017-05-31", "2017-06-16", "2017-06-20", "2017-09-18", "2…

Rows: 207
Columns: 3
$ subid       <dbl> 1, 2, 3, 5, 6, 7, 9, 10, 11, 15, 16, 17, 18, 19, 20, 21, 2…
$ start_study <dttm> 2017-03-02 04:00:00, 2017-03-24 04:00:00, 2017-03-22 04:0…
$ end_study   <dttm> 2017-05-31 04:00:00, 2017-06-16 04:00:00, 2017-06-20 04:0…

Rows: 1,076
Columns: 14
$ subid            <dbl> 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 7, 7, 7, 7,…
$ lapse_start_date <chr> "04-01-2017", "04-03-2017", "04-29-2017", "05-06-2017…
$ lapse_start_time <time> 19:00:00, 18:00:00, 20:00:00, 20:00:00, 11:00:00, 20…
$ lapse_start      <dttm> 2017-04-01 19:00:00, 2017-04-03 18:00:00, 2017-04-29…
$ lapse_end_date   <chr> "04-01-2017", "04-03-2017", "04-29-2017", "05-06-2017…
$ lapse_end_time   <time> 19:00:00, 21:00:00, 22:00:00, 21:00:00, 11:00:00, 21…
$ lapse_end        <dttm> 2017-04-02 00:00:00, 2017-04-04 02:00:00, 2017-04-30…
$ duration         <dbl> 0, 3, 2, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 4,…
$ source           <chr> "ema", "ema", "ema", "ema", "ema", "ema", "ema", "ema…
$ ema_1_6          <chr> "America/Chicago", "America/Chicago", "America/Chicag…
$ response_id      <chr> "R_1Hqy98ViZigIUr3", "R_3oTRLcP1UfdS1bb", "R_6VFN7egq…
$ ema_end          <dttm> 2017-06-16 14:18:40, 2017-06-16 14:18:40, 2017-06-16…
$ exclude       

In [ ]:
labels <- visit_dates |> 
  drop_na() |> 
  rowwise() |> 
  mutate(lapse_onset = list(seq.POSIXt(start_study, end_study, by = "day"))) |> 
  unnest(lapse_onset) |>
  select(subid, lapse_onset) |> 
  glimpse()

Rows: 13,619
Columns: 2
$ subid       <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1…
$ lapse_onset <dttm> 2017-03-02 04:00:00, 2017-03-03 04:00:00, 2017-03-04 04:0…

In [ ]:
labels <- labels |> 
  full_join(lapses |> select(subid, lapse_start), by = "subid") |> 
  mutate(
    lapse_end = lapse_onset + days(1),
    label = if_else(lapse_start >= lapse_onset & lapse_start <= lapse_end,
                    "yes", "no", missing = "no")
  ) |> 
  group_by(subid, lapse_onset) |> 
  summarize(label = if_else(any(label == "yes"), "yes", "no")) |> 
  ungroup() |> 
  glimpse()

Warning in full_join(labels, select(lapses, subid, lapse_start), by = "subid"): Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 91 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.

`summarise()` has grouped output by 'subid'. You can override using the
`.groups` argument.

Rows: 13,619
Columns: 3
$ subid       <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1…
$ lapse_onset <dttm> 2017-03-02 04:00:00, 2017-03-03 04:00:00, 2017-03-04 04:0…
$ label       <chr> "no", "no", "no", "no", "no", "no", "no", "no", "no", "no"…

## Write-out csv

In [ ]:
labels |> 
  write_csv(here(path_messages, "lapses.csv")) |> 
  glimpse()

Rows: 13,619
Columns: 3
$ subid       <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1…
$ lapse_onset <dttm> 2017-03-02 04:00:00, 2017-03-03 04:00:00, 2017-03-04 04:0…
$ label       <chr> "no", "no", "no", "no", "no", "no", "no", "no", "no", "no"…